# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI協助）
- Date first created: 20261019
- Background: 黃裕庭新增2022仁政里里長生日／政府性別及歷史學歷級別支持；陳傑麟、陳泓維新增2022議員身分連結，缺學歷者補當年申報級別。人物網頁數量不增加。
- Unit of observation: master/council/pending/review為候選人；fields為候選人×欄位；evidence為候選人×欄位×來源。
- Sample period: 2026候選人登記框架，2014–2022歷史來源與2026自述按來源分開。
- Known issues: 730人的這幾類網頁缺口尚未補齊；151個歷史身分待查。703位2022議員申報生日，跨年／跨公職政府生日支持聯集707人；明載性別聯集707人仍有不同來源層級。沒有找到不填首次=1。魏造文性別與22位縣市長生日仍缺。社群87人201組入口，不代表完整貼文可取得；Meta與窗口暫緩。
- Definition for each variable: 沿用[第七輪資料字典與欄位schema](identity_round7_20261019_documentation.ipynb)。birth_date_supported_history/sex_supported_history只含議員歷史連結；birth_date_additional_official記跨公職／不同年代；sex_additional_source_value另存新增來源；additional_historical_education_cec為官方當年學歷級別；education_raw補空白但不虛構學校。review的identity_anchor記具體職務／年份／地理碼橋接，history_file/row指定官方列；evidence按候選人×欄位×來源保存。candidate_key與field組合鍵依各檔定義驗證。
- Data path / Downstream analysis input: 下方6個精確保存檔；角色review_inventory/evidence，尚非最終分析樣本。
- Source data: 處理Notebook PINS固定4個pre_cross_office_round8快照、官方歷史CSV、行政區碼及來源manifest／人物頁；全清單與hash在下方SOURCE_PINS。
- Processing code: [cross_office_round8_20261019.ipynb](../code/cross_office_round8_20261019.ipynb)。
- Role: review inventory / evidence。
- Verification status: 下方讀回檢查存在、schema、鍵、SHA及必要不變欄位；不覆寫資料。
- SHA-256: 各檔如下。
- `data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv`：1583列／34欄，SHA-256 `4fba9288b89aa417734ea686b09ccad5eed147b47162265c799319bd025fa926`。
- `data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv`：31660列／11欄，SHA-256 `80d9ccb7ea2cb1e5adfd3488f0b3d16581b067311d9e230f3daee37cf7b059ad`。
- `data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv`：1502列／50欄，SHA-256 `980d3f2c01d304ed4c9c94128602fb2f55ae8429516b58e92ddbe13886240831`。
- `data/processed/council_history_pending_identity_current.csv`：151列／18欄，SHA-256 `0e96766f01de00a4a06674ff550c4dc1cd57361e4f9b5c07c7328cce3f740f29`。
- `data/processed/council_identity_review_round8_20261019.csv`：3列／12欄，SHA-256 `bb6679db0d1e7e23713b45267999cea2b2a4139c45a953f9854799ee7ee89b29`。
- `data/processed/candidate_evidence_round8_20261019.csv`：13列／9欄，SHA-256 `94058c0bb7bf32312bcfc1e02a87d628a52f44d529b402348b30f74d3ed1302d`。

第九輪接續：本輪3份current整合表封存至pre_municipal_round9快照；舊程式不覆寫最新背景表。待核對歷史工作單本輪未改，仍151人。

In [1]:
from pathlib import Path
import pandas as pd,json,hashlib,csv
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
SOURCE_PINS={'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_master_current.csv': '0d2d2620f62cde3abaf691ce843925231bc84f95362562b16794d74aadf022e9', 'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_fields_current.csv': 'e2e4528410ee10272ddef071f431e42334e14891255277a1a44dce2203ef9b22', 'data/processed/snapshots/pre_cross_office_round8_20261019/council_background_20261019.csv': '47ad3b02ee159bd1a6e7aa9114750ee6bde8d4c4380eacac24b41f116deba410', 'data/processed/snapshots/pre_cross_office_round8_20261019/council_history_pending_identity_current.csv': 'abf997cfa9ceac55cd7636db7950cfa934cb5d553183f5ff05d6181bc9bd72e0', 'data/raw/cec_all_offices/107_elcand.csv': 'af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce7441ef7cb52d02f1b9', 'data/raw/cec_all_offices/110_elcand.csv': 'b323d6713df428b57161e3b1b6799ebf25d679ef9948e8fe4fbb3fa0d7cfbc73', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cec_all_offices/095_elcand.csv': '2106bc9400f7d64784595799d2404496186cffaa56d82b14554d172d781f1b60', 'data/raw/cec_all_offices/073_elcand.csv': '3d5391d0276093fb4721778d84b8d36c4569457d1d865ee14d45805939d9e5bd', 'data/raw/cec_all_offices/087_elcand.csv': '58616de64001f72a3f16be830a15aceb42cecdc63b239ce234c13c3ba408e307', 'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv': '4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79', 'data/raw/cross_office_round8_20261019/manifest.json': '6c3b1612b234da8a0ab7e799fe7367bc6ebaae9ef48f190144fbb09ed4124022', 'data/raw/social_followup_round6_20261019/manifest.json': 'f5471133039402a9e21143f23d30b70b645ddf3406805e19f78764f5b2063746', 'data/raw/kmtks_web_20261019/manifest.json': 'a519c5d80a86591bec6da77b7f031e56c5c10fa9e79245e5e1b74332a4709fb8', 'data/raw/social_followup_round6_20261019/b39ba23e64e7bb9ca8b3.html': 'c0a7c18159f0dc6e66dfcca6757183d0eb26f66487871cd5fb635c1c4e72bdd6', 'data/raw/social_followup_round6_20261019/0277960ed4f266884bad.html': 'f31d0165ee321abb26ce8b9cb6816b9dbbd4655fc9a7f9ddc309a75bcbc12c90', 'data/raw/kmtks_web_20261019/05.txt': '410ff727927bf36c12bc505782951ead7cfb91b88b0e1d28e4802249c7f99b26'}
EXPECTED={'data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv': {'rows': 1583, 'columns': 34, 'key': ['candidate_key'], 'sha256': '4fba9288b89aa417734ea686b09ccad5eed147b47162265c799319bd025fa926'}, 'data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv': {'rows': 31660, 'columns': 11, 'key': ['candidate_key', 'field'], 'sha256': '80d9ccb7ea2cb1e5adfd3488f0b3d16581b067311d9e230f3daee37cf7b059ad'}, 'data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv': {'rows': 1502, 'columns': 50, 'key': ['candidate_key'], 'sha256': '980d3f2c01d304ed4c9c94128602fb2f55ae8429516b58e92ddbe13886240831'}, 'data/processed/council_history_pending_identity_current.csv': {'rows': 151, 'columns': 18, 'key': ['candidate_key'], 'sha256': '0e96766f01de00a4a06674ff550c4dc1cd57361e4f9b5c07c7328cce3f740f29'}, 'data/processed/council_identity_review_round8_20261019.csv': {'rows': 3, 'columns': 12, 'key': ['candidate_key'], 'sha256': 'bb6679db0d1e7e23713b45267999cea2b2a4139c45a953f9854799ee7ee89b29'}, 'data/processed/candidate_evidence_round8_20261019.csv': {'rows': 13, 'columns': 9, 'key': [], 'sha256': '94058c0bb7bf32312bcfc1e02a87d628a52f44d529b402348b30f74d3ed1302d'}}
for p,h in SOURCE_PINS.items():assert sha(p)==h,p
D={}
for p,s in EXPECTED.items():
 assert sha(p)==s['sha256'],p
 d=pd.read_csv(ROOT/p,dtype=str,keep_default_na=False);assert d.shape==(s['rows'],s['columns'])
 if s['key']:assert not d.duplicated(s['key']).any()
 D[p]=d
print('Fixed inputs and saved outputs checked:',len(SOURCE_PINS),len(D))

Fixed inputs and saved outputs checked: 18 6


## Change for Your Own Data
只讀固定版本與hash。

## Summary

### Sample

In [2]:
for p,d in D.items():print(p);display(d.head(10))

data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv


,candidate_key,candidate_name,office_type,county_city,electoral_district,party_registration_raw,registration_source,registration_source_file,registration_source_page,qualification_inherited,...,Threads_status,posts_Facebook_status,posts_Instagram_status,posts_Threads_status,followers_status,policy_text_status,visual_status,political_family_status,administrative_experience_status,qualification_status
0,2026-C1-001,郭璽,直轄市長,臺北市,,台灣麻將最大黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
1,2026-C1-002,沈伯洋,直轄市長,臺北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
2,2026-C1-003,蕭文乾,直轄市長,臺北市,,臺灣SoR無法黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
3,2026-C1-004,唐新民,直轄市長,臺北市,,三勢團結促進聯盟,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
4,2026-C1-005,蔣萬安,直轄市長,臺北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
5,2026-C1-006,林志成,直轄市長,臺北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
6,2026-C1-007,蘇巧慧,直轄市長,新北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
7,2026-C1-008,李四川,直轄市長,新北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
8,2026-C1-009,蘇輝湟,直轄市長,新北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
9,2026-C1-010,張善政,直轄市長,桃園市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對


data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv


,candidate_key,candidate_name,office_type,county_city,field,status,existing_value,evidence,note,next_source,updated_date
0,2026-C1-001,郭璽,直轄市長,臺北市,sex,已有資料，待來源複核,男,https://db.cec.gov.tw/,dob_corroborated_by_media,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
1,2026-C1-001,郭璽,直轄市長,臺北市,birth_date,已有資料，待來源複核,1958-08-31,https://db.cec.gov.tw/,CEC_name_and_birth_date_corroborated_by_CNA,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
2,2026-C1-001,郭璽,直轄市長,臺北市,education,已有資料，待來源複核,大學,,歷史公報或媒體補充，非2026學歷認證,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
3,2026-C1-001,郭璽,直轄市長,臺北市,incumbency,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
4,2026-C1-001,郭璽,直轄市長,臺北市,first_time_this_office,既有搜尋後仍缺，待補查,,,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
5,2026-C1-001,郭璽,直轄市長,臺北市,first_time_any_office,已有資料，待來源複核,0,https://db.cec.gov.tw/,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
6,2026-C1-001,郭璽,直轄市長,臺北市,profile,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
7,2026-C1-001,郭璽,直轄市長,臺北市,website,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
8,2026-C1-001,郭璽,直轄市長,臺北市,Facebook,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19
9,2026-C1-001,郭璽,直轄市長,臺北市,Instagram,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19


data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv


...

,candidate_key,candidate_name,county_city,electoral_district,same_district_number,birth_date_history_candidate,identity_status,profile_evidence_present,history_name,history_county,history_district,history_birth_roc,history_sex,history_education,history_file,history_row,history_archive_member,source_sha256
0,2026|直轄市議員|臺北市第2選舉區|吳欣岱,吳欣岱,臺北市,臺北市第2選舉區,True,1987-07-01,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,True,吳欣岱,臺北市,2,0760701,女,大學,data/raw/cec_all_offices/107_elcand.csv,34,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
1,2026|直轄市議員|臺北市第3選舉區|林柏勛,林柏勛,臺北市,臺北市第3選舉區,True,1983-11-15,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,林柏勛,臺北市,3,0721115,男,碩士,data/raw/cec_all_offices/107_elcand.csv,43,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
2,2026|直轄市議員|臺北市第5選舉區|吳怡萱,吳怡萱,臺北市,臺北市第5選舉區,True,1989-08-17,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,吳怡萱,臺北市,5,0780817,女,大學,data/raw/cec_all_offices/107_elcand.csv,75,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
3,2026|直轄市議員|臺北市第6選舉區|陳聖文,陳聖文,臺北市,臺北市第6選舉區,True,1998-05-06,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,陳聖文,臺北市,6,0870506,男,碩士,data/raw/cec_all_offices/107_elcand.csv,112,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
4,2026|直轄市議員|新北市第2選舉區|張昭隆,張昭隆,新北市,新北市第2選舉區,False,1966-10-22,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,張昭隆,新北市,8,0551022,男,大學,data/raw/cec_all_offices/107_elcand.csv,318,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
5,2026|直轄市議員|新北市第4選舉區|董俊良,董俊良,新北市,新北市第4選舉區,True,1970-07-30,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,董俊良,新北市,4,0590730,男,高中(職),data/raw/cec_all_offices/107_elcand.csv,276,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
6,2026|直轄市議員|新北市第4選舉區|陳宛毓,陳宛毓,新北市,新北市第4選舉區,True,1988-04-18,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,True,陳宛毓,新北市,4,0770418,女,大學,data/raw/cec_all_offices/107_elcand.csv,266,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
7,2026|直轄市議員|新北市第5選舉區|戴瑋姗,戴瑋姗,新北市,新北市第5選舉區,True,1990-01-23,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,戴瑋姗,新北市,5,0790123,女,大學,data/raw/cec_all_offices/107_elcand.csv,282,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
8,2026|直轄市議員|新北市第6選舉區|張志豪,張志豪,新北市,新北市第6選舉區,True,1981-08-24,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,張志豪,新北市,6,0700824,男,碩士,data/raw/cec_all_offices/107_elcand.csv,307,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...
9,2026|直轄市議員|新北市第7選舉區|羅文崇,羅文崇,新北市,新北市第7選舉區,True,1964-07-08,同名同縣市歷史命中，身分仍待核對；未填入已核實欄,False,羅文崇,新北市,7,0530708,男,碩士,data/raw/cec_all_offices/107_elcand.csv,312,votedata/votedata/voteData/2022-111年地方公職人員選舉/T...,af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce...


data/processed/council_identity_review_round8_20261019.csv


,candidate_key,candidate_name,decision,birth_date,sex,history_file,history_row,identity_url,identity_file,identity_anchor,first_time_council_conclusion,prior_official_files
0,2026|直轄市議員|高雄市第8選舉區|黃裕庭,黃裕庭,supported_cross_office_village_history,1981-01-01,男,data/raw/cec_all_offices/112_elcand.csv,8012,https://2026.kmtks.org.tw/Council/huang-yu-ting,data/raw/kmtks_web_20261019/05.txt,高雄苓雅仁政里第3、4屆里長，官方行政區碼逐層一致,unknown; village election does not establish p...,
1,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,supported_2022_history_cross_year_bridge,1987-05-08,男,data/raw/cec_all_offices/107_elcand.csv,602,https://youth.pfpnext.com/2026/candidate-detai...,data/raw/social_followup_round6_20261019/b39ba...,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,nonfirst based on 2022 council registration,data/raw/cec_all_offices/095_elcand.csv
2,2026|縣市議員|宜蘭縣第12選舉區|陳傑麟,陳傑麟,supported_2022_history_cross_year_bridge,1965-11-08,男,data/raw/cec_all_offices/110_elcand.csv,2,https://youth.pfpnext.com/2026/candidate-detai...,data/raw/social_followup_round6_20261019/02779...,當屆明載宜蘭第18、19屆議員；2014、2018同選區當選紀錄及2022完整生日／性別一致,nonfirst based on 2022 council registration,data/raw/cec_all_offices/073_elcand.csv | data...


data/processed/candidate_evidence_round8_20261019.csv


,candidate_key,candidate_name,field,value,source_file,source_row,source_url,evidence_status,checked_date
0,2026|直轄市議員|高雄市第8選舉區|黃裕庭,黃裕庭,birth_date,1981-01-01,data/raw/cec_all_offices/112_elcand.csv,8012,https://db.cec.gov.tw/,2022里長名冊；對應當屆仁政里里長自述；非2026戶籍／學位認證,2026-10-19
1,2026|直轄市議員|高雄市第8選舉區|黃裕庭,黃裕庭,sex,男,data/raw/cec_all_offices/112_elcand.csv,8012,https://db.cec.gov.tw/,2022里長名冊；對應當屆仁政里里長自述；非2026戶籍／學位認證,2026-10-19
2,2026|直轄市議員|高雄市第8選舉區|黃裕庭,黃裕庭,education,碩士,data/raw/cec_all_offices/112_elcand.csv,8012,https://db.cec.gov.tw/,2022里長名冊；對應當屆仁政里里長自述；非2026戶籍／學位認證,2026-10-19
3,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,birth_date,1987-05-08,data/raw/cec_all_offices/107_elcand.csv,602,https://db.cec.gov.tw/,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,2026-10-19
4,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,sex,男,data/raw/cec_all_offices/107_elcand.csv,602,https://db.cec.gov.tw/,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,2026-10-19
5,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,education,高中(職),data/raw/cec_all_offices/107_elcand.csv,602,https://db.cec.gov.tw/,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,2026-10-19
6,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,first_time_this_office,0,data/raw/cec_all_offices/107_elcand.csv,602,https://db.cec.gov.tw/,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,2026-10-19
7,2026|直轄市議員|桃園市第2選舉區|陳泓維,陳泓維,first_time_any_office,0,data/raw/cec_all_offices/107_elcand.csv,602,https://db.cec.gov.tw/,當屆明載2020桃園第1選區立委參選；2020與2022生日、性別及學歷級別一致,2026-10-19
8,2026|縣市議員|宜蘭縣第12選舉區|陳傑麟,陳傑麟,birth_date,1965-11-08,data/raw/cec_all_offices/110_elcand.csv,2,https://db.cec.gov.tw/,當屆明載宜蘭第18、19屆議員；2014、2018同選區當選紀錄及2022完整生日／性別一致,2026-10-19
9,2026|縣市議員|宜蘭縣第12選舉區|陳傑麟,陳傑麟,sex,男,data/raw/cec_all_offices/110_elcand.csv,2,https://db.cec.gov.tw/,當屆明載宜蘭第18、19屆議員；2014、2018同選區當選紀錄及2022完整生日／性別一致,2026-10-19


### Summary Stats

In [3]:
for p,d in D.items():
 print(p);display(pd.DataFrame({'type':d.dtypes.astype(str),'empty':d.eq('').sum()}))
c=D['data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv'];display(c.loc[c.candidate_name.isin(['黃裕庭','陳泓維','陳傑麟']),['candidate_name','birth_date_supported_history','birth_date_additional_official','education_raw','first_time_this_office_supported']])

data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
office_type,str,0
county_city,str,0
electoral_district,str,81
party_registration_raw,str,0
registration_source,str,0
registration_source_file,str,0
registration_source_page,str,0
qualification_inherited,str,0


data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
office_type,str,0
county_city,str,0
field,str,0
status,str,0
existing_value,str,25211
evidence,str,24229
note,str,23931
next_source,str,0


data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv


...

data/processed/council_identity_review_round8_20261019.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
decision,str,0
birth_date,str,0
sex,str,0
history_file,str,0
history_row,str,0
identity_url,str,0
identity_file,str,0
identity_anchor,str,0


data/processed/candidate_evidence_round8_20261019.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
field,str,0
value,str,0
source_file,str,0
source_row,str,0
source_url,str,0
evidence_status,str,0
checked_date,str,0


,candidate_name,birth_date_supported_history,birth_date_additional_official,education_raw,first_time_this_office_supported
234,陳泓維,1987-05-08,1987-05-08,高中(職),0
569,黃裕庭,,1981-01-01,國立高雄師範大學教育研究所碩士,
1150,陳傑麟,1965-11-08,,碩士,0


### Assertions

In [4]:
before=pd.read_csv(ROOT/'data/processed/snapshots/pre_cross_office_round8_20261019/council_background_20261019.csv',dtype=str,keep_default_na=False)
assert c[['candidate_key','party_registration_raw','incumbency_2026']].equals(before[['candidate_key','party_registration_raw','incumbency_2026']])
assert c.birth_date_supported_history.ne('').sum()==703
assert (c.birth_date_supported_history.ne('')|c.birth_date_additional_official.ne('')).sum()==707
assert (c.sex_supported_history.ne('')|c.sex_additional_source_value.ne('')).sum()==707
assert c.first_time_any_office_supported.eq('0').sum()==712 and c.first_time_this_office_supported.eq('0').sum()==705
assert len(D['data/processed/council_history_pending_identity_current.csv'])==151
assert c.loc[c.candidate_name.eq('黃裕庭'),'first_time_this_office_supported'].eq('').all()
assert (c.moi_profile_url.eq('')&c.campaign_profile_url.eq('')&c.additional_profile_url.eq('')).sum()==730
print('PASS: saved identity/schema/key/hash, source bridges, counts, village-vs-council distinction, party/incumbency preserved.')

PASS: saved identity/schema/key/hash, source bridges, counts, village-vs-council distinction, party/incumbency preserved.


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。